In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import sklearn
from sklearn.model_selection import train_test_split
from optbinning import OptimalBinning
from sklearn.utils import check_array
def safe_check_array(X, **kwargs):
    if "force_all_finite" in kwargs and sklearn.__version__ >= "1.6":
        kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
    return check_array(X, **kwargs)

RAW_DATA = "data/raw_data"
FI_DATA = "data/imp_features"
SEED = 42

BAD_STATUSES = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

TARGET = 'default_status'

In [2]:
acc_df = pd.read_csv(os.path.join(RAW_DATA, os.path.join("accepted_2007_to_2018q4.csv", "accepted_2007_to_2018q4.csv"))).sample(n=500000, random_state=SEED)
print(f"Shape of acc_df_org {acc_df.shape}")

Shape of acc_df_org (500000, 151)


## Split data Train | Validation | OOT
1. Train: Training Data (70%)
2. Validation: Using for risk segment calculation (15%)
3. OOT: Testing the (15%)

In [3]:
acc_df[TARGET] = acc_df['loan_status'].apply(lambda x: 1 if x in BAD_STATUSES else 0)
acc_df = acc_df.drop(['loan_status', 'id'], axis='columns')

print(f" Null values before | issue_d {acc_df['issue_d'].isnull().sum()}")
acc_df = acc_df[acc_df['issue_d'].notnull()]
acc_df['issue_d'] = pd.to_datetime(acc_df['issue_d'])
acc_df = acc_df.sort_values(
    by="issue_d", ascending=True ).reset_index(drop=True)
print(f" Null values after | issue_d {acc_df['issue_d'].isnull().sum()}")

n = len(acc_df)
train_end = int(n * 0.70)
test_end = int(n * 0.85)

train = acc_df.iloc[:train_end]
test = acc_df.iloc[train_end:test_end]
oot = acc_df.iloc[test_end:]

print(f"Shape of  train {train.shape} | test {test.shape} | oot {oot.shape}")
print(f"Bad rate           : {acc_df[TARGET].value_counts(normalize=True)[1].round(2) * 100} %")

 Null values before | issue_d 8
 Null values after | issue_d 0
Shape of  train (349994, 150) | test (74999, 150) | oot (74999, 150)
Bad rate           : 12.0 %


## 1. Fill Rate Analysis

In [4]:
FR_THRESHOLD = 5

fill_rate_report = pd.DataFrame({
    "COLUMN_NAME": acc_df.columns,
    "TOTAL_ROWS": len(acc_df),
    "NON_NULL_COUNT": acc_df.notnull().sum().values,
    "NULL_COUNT": acc_df.isnull().sum().values
})

fill_rate_report["FILL_RATE"] = (
    fill_rate_report["NON_NULL_COUNT"]
    / fill_rate_report["TOTAL_ROWS"]
    * 100
).round(2)

fill_rate_report["DROP_FLAG"] = np.where(
    fill_rate_report["FILL_RATE"] < FR_THRESHOLD,
    "DROP",
    "KEEP"
)

fill_rate_report = fill_rate_report.sort_values(
    by="FILL_RATE",
    ascending=True
).reset_index(drop=True)

print(f"Total Features                  : {acc_df.shape[1]}")
print(f"Features Below {FR_THRESHOLD}% FR : {(fill_rate_report['DROP_FLAG'] == 'DROP').sum()}")
print(f"Features To Keep                : {(fill_rate_report['DROP_FLAG'] == 'KEEP').sum()}")

fill_rate_report.head(20)

Total Features                  : 150
Features Below 5% FR : 34
Features To Keep                : 116


,COLUMN_NAME,TOTAL_ROWS,NON_NULL_COUNT,NULL_COUNT,FILL_RATE,DROP_FLAG
0,member_id,499992,0,499992,0.00,DROP
1,orig_projected_additional_accrued_interest,499992,1825,498167,0.37,DROP
2,hardship_end_date,499992,2353,497639,0.47,DROP
3,hardship_type,499992,2353,497639,0.47,DROP
4,hardship_payoff_balance_amount,499992,2353,497639,0.47,DROP
5,hardship_loan_status,499992,2353,497639,0.47,DROP
6,hardship_dpd,499992,2353,497639,0.47,DROP
7,hardship_last_payment_amount,499992,2353,497639,0.47,DROP
8,hardship_amount,499992,2353,497639,0.47,DROP
9,payment_plan_start_date,499992,2353,497639,0.47,DROP


In [5]:
keep_cols = fill_rate_report.loc[
    fill_rate_report["DROP_FLAG"].isin(['KEEP']),
    "COLUMN_NAME"
].tolist()

assert TARGET in keep_cols, "dropping target column"

acc_df_fr = acc_df[keep_cols]
train_fr = train[keep_cols]
test_fr = test[keep_cols]
oot_fr = oot[keep_cols]

print(acc_df.shape, "->", acc_df_fr.shape)
print(train.shape, "->", train_fr.shape)
print(test.shape, "->", test_fr.shape)
print(oot.shape, "->", oot_fr.shape)

(499992, 150) -> (499992, 116)
(349994, 150) -> (349994, 116)
(74999, 150) -> (74999, 116)
(74999, 150) -> (74999, 116)


## 2. Zero / Near-Zero Variance Filter
- Remove variables that barely change.

In [6]:
VAR_THRESHOLD = 0.99

nzv_report = []

for col in acc_df_fr.columns:

    vc = acc_df_fr[col].value_counts(dropna=False, normalize=True)
    top_freq = vc.iloc[0] if len(vc) else np.nan

    nzv_report.append({
        "COLUMN_NAME": col,
        "N_UNIQUE": acc_df_fr[col].nunique(dropna=False),
        "TOP_FREQ_PCT": round(top_freq * 100, 2),
        "DROP_FLAG": (
            "DROP"
            if (
                acc_df_fr[col].nunique(dropna=False) <= 1
                or top_freq >= VAR_THRESHOLD
            )
            else "KEEP"
        )
    })

nzv_report = pd.DataFrame(nzv_report).sort_values(
    "TOP_FREQ_PCT",
    ascending=False
)

print(f"Total Features       : {len(nzv_report)}")
print(f"Near-Zero Variance   : {(nzv_report['DROP_FLAG'] == 'DROP').sum()}")
print(f"Features To Keep     : {(nzv_report['DROP_FLAG'] == 'KEEP').sum()}")

nzv_report.head(20)

Total Features       : 116
Near-Zero Variance   : 6
Features To Keep     : 110


,COLUMN_NAME,N_UNIQUE,TOP_FREQ_PCT,DROP_FLAG
80,policy_code,1,100.00,DROP
72,pymnt_plan,2,99.97,DROP
94,hardship_flag,2,99.97,DROP
96,delinq_amnt,913,99.69,DROP
82,acc_now_delinq,8,99.61,DROP
65,chargeoff_within_12_mths,11,99.23,DROP
114,debt_settlement_flag,2,98.48,KEEP
66,collections_12_mths_ex_med,11,98.35,KEEP
95,tax_liens,28,97.14,KEEP
39,num_tl_30dpd,6,96.66,KEEP


In [7]:
keep_cols = nzv_report.loc[
    nzv_report["DROP_FLAG"] == "KEEP",
    "COLUMN_NAME"
].tolist()

assert TARGET in keep_cols, "dropping target column"

acc_df_nzv = acc_df_fr[keep_cols]
train_nzv = train_fr[keep_cols]
test_nzv = test_fr[keep_cols]
oot_nzv = oot_fr[keep_cols]

print(acc_df_fr.shape, "->", acc_df_nzv.shape)
print(train_fr.shape, "->", train_nzv.shape)
print(test_fr.shape, "->", test_nzv.shape)
print(oot_fr.shape, "->", oot_nzv.shape)

(499992, 116) -> (499992, 110)
(349994, 116) -> (349994, 110)
(74999, 116) -> (74999, 110)
(74999, 116) -> (74999, 110)


"## 3. WoE / IV Computation

In [8]:
IV_SAMPLE_SIZE = 100000
sample_frac = min(
    1.0,
    IV_SAMPLE_SIZE / len(acc_df_nzv)
)

iv_sample, _ = train_test_split(
    acc_df_nzv,
    train_size=sample_frac,
    stratify=acc_df_nzv[TARGET],
    random_state=42
)

print(f"Original Shape : {acc_df_nzv.shape}")
print(f"Sample Shape   : {iv_sample.shape}")

print(f"Original Bad Rate : ", f"{acc_df_nzv[TARGET].mean():.4f}")
print(f"Sample Bad Rate   : " f"{iv_sample[TARGET].mean():.4f}")

IV_THRESHOLD = 0.2
iv_report = []

for col in iv_sample.columns:
    try:
        X = iv_sample[col]
        y = iv_sample[TARGET]

        bins = OptimalBinning(
            name=col,
            dtype= 'numerical' if pd.api.types.is_numeric_dtype(X) else 'categorical'
        )

        bins.fit(X, y)
        iv = bins.binning_table.build()["IV"].sum()
        iv_report.append({
            "COLUMN_NAME": col,
            "IV":iv,
            "DROP_FLAG": "KEEP" if iv >= IV_THRESHOLD else "DROP"
        })

    except Exception as e:
        print(e)
        iv_report.append({
            "COLUMN_NAME": col,
            "IV": np.nan,
            "DROP_FLAG": "DROP"
        })


iv_report = pd.DataFrame(iv_report).sort_values(
    "IV",
    ascending=False
)

print(f"Total Features      : {len(iv_report)}")
print(f"Features To Keep    : {(iv_report['DROP_FLAG'] == 'KEEP').sum()}")
print(f"Features To Drop    : {(iv_report['DROP_FLAG'] == 'DROP').sum()}")

iv_report.head(20)

Original Shape : (499992, 110)
Sample Shape   : (100000, 110)
Original Bad Rate :  0.1197
Sample Bad Rate   : 0.1197
Total Features      : 110
Features To Keep    : 34
Features To Drop    : 76


,COLUMN_NAME,IV,DROP_FLAG
77,emp_title,11.576579,KEEP
84,last_fico_range_low,8.616136,KEEP
85,last_fico_range_high,8.616136,KEEP
29,out_prncp,7.191399,KEEP
30,out_prncp_inv,7.191333,KEEP
53,last_pymnt_d,5.427518,KEEP
31,next_pymnt_d,4.938550,KEEP
104,last_pymnt_amnt,3.850415,KEEP
28,last_credit_pull_d,2.541672,KEEP
87,total_rec_prncp,1.905170,KEEP


In [9]:
keep_cols = iv_report.loc[
    iv_report["DROP_FLAG"] == "KEEP",
    "COLUMN_NAME"
].tolist()

if TARGET not in keep_cols:
    keep_cols.append(TARGET)

acc_df_bin = acc_df_nzv[keep_cols]
train_bin = train_nzv[keep_cols]
test_bin = test_nzv[keep_cols]
oot_bin = oot_nzv[keep_cols]

print(acc_df_nzv.shape, "->", acc_df_bin.shape)
print(train_nzv.shape, "->", train_bin.shape)
print(test_nzv.shape, "->", test_bin.shape)
print(oot_nzv.shape, "->", oot_bin.shape)

(499992, 110) -> (499992, 35)
(349994, 110) -> (349994, 35)
(74999, 110) -> (74999, 35)
(74999, 110) -> (74999, 35)


## POST Disbursal Data Identification & Removal

In [11]:
post_loan_cols = [
    # Payment history
    "last_pymnt_d",
    "next_pymnt_d",
    "last_pymnt_amnt",
    "total_rec_prncp",
    "total_pymnt",
    "total_pymnt_inv",

    # Loan performance / current status
    "out_prncp",
    "out_prncp_inv",

    # Information updated during servicing
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
]

keep_cols = [col for col in acc_df_bin.columns if col not in post_loan_cols]

if TARGET not in keep_cols:
    keep_cols.append(TARGET)

acc_df_op = acc_df_bin[keep_cols]
train_op = train_bin[keep_cols]
test_op = test_bin[keep_cols]
oot_op = oot_bin[keep_cols]

print(acc_df_bin.shape, "->", acc_df_op.shape)
print(train_bin.shape, "->", train_op.shape)
print(test_bin.shape, "->", test_op.shape)
print(oot_bin.shape, "->", oot_op.shape)

(499992, 35) -> (499992, 24)
(349994, 35) -> (349994, 24)
(74999, 35) -> (74999, 24)
(74999, 35) -> (74999, 24)


In [12]:
drop_funnel = pd.DataFrame({
    "STAGE": [
        "Initial Features",
        "After Fill Rate",
        "After NZV",
        "After IV",
        "After Drop Outputs"
    ],
    "FEATURE_COUNT": [
        acc_df.shape[1] - 1,
        acc_df_fr.shape[1] - 1,
        acc_df_nzv.shape[1] - 1,
        acc_df_bin.shape[1] - 1, 
        acc_df_op.shape[1] - 1
    ]
})

drop_funnel["FEATURES_DROPPED"] = (
    drop_funnel["FEATURE_COUNT"]
    .shift(1)
    .sub(drop_funnel["FEATURE_COUNT"])
    .fillna(0)
    .astype(int)
)

drop_funnel["DROP_PCT"] = (
    drop_funnel["FEATURES_DROPPED"]
    / drop_funnel["FEATURE_COUNT"].shift(1)
    * 100
).fillna(0).round(2)

drop_funnel["SURVIVAL_PCT"] = (
    drop_funnel["FEATURE_COUNT"]
    / drop_funnel.loc[0, "FEATURE_COUNT"]
    * 100
).round(2)

print(drop_funnel)

                STAGE  FEATURE_COUNT  FEATURES_DROPPED  DROP_PCT  SURVIVAL_PCT
0    Initial Features            149                 0      0.00        100.00
1     After Fill Rate            115                34     22.82         77.18
2           After NZV            109                 6      5.22         73.15
3            After IV             34                75     68.81         22.82
4  After Drop Outputs             23                11     32.35         15.44


## Saving files

In [13]:
acc_df_op.to_csv(os.path.join(FI_DATA, 'full_df_fi.csv'), index=False)
train_op.to_csv(os.path.join(FI_DATA, 'train_fi.csv'), index=False)
test_op.to_csv(os.path.join(FI_DATA, 'test_fi.csv'), index=False)
oot_op.to_csv(os.path.join(FI_DATA, 'oot_fi.csv'), index=False)